# Titanic Survival Classification

Predict passenger survival from tabular features using classical machine learning.

This study compares a **Decision Tree** and **k-Nearest Neighbors** classifier after careful preprocessing of mixed numeric and categorical fields.

| Item | Detail |
|------|--------|
| Task | Binary classification (`survived`) |
| Input | Passenger attributes (class, sex, age, fare, family counts, …) |
| Models | Decision Tree, KNN |
| Metrics | Accuracy, F1-score, ROC-AUC |
| Split | Stratified 60 / 20 / 20 train / validation / test |


## Setup


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_curve,
    auc,
    classification_report,
)

REPO_ROOT = Path("../../..").resolve()
if not (REPO_ROOT / "common" / "portfolio_style.py").exists():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "common" / "portfolio_style.py").exists():
            REPO_ROOT = candidate
            break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from common.portfolio_style import (
    PALETTE,
    CLASS_COLORS,
    apply_mpl_style,
    plot_confusion as _plot_confusion,
)

apply_mpl_style()

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

CLASS_NAMES = {0: "Did not survive", 1: "Survived"}


def plot_confusion(y_true, y_pred, title, save_as=None):
    path = FIGURES_DIR / save_as if save_as else None
    return _plot_confusion(
        y_true,
        y_pred,
        title,
        class_names=[CLASS_NAMES[0], CLASS_NAMES[1]],
        save_as=path,
    )


def plot_roc(y_true, y_prob, title, save_as=None):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ax.plot(fpr, tpr, color=PALETTE["med_blue"], linewidth=2, label=f"AUC = {roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], color=PALETTE["slate"], linestyle="--", linewidth=1)
    ax.fill_between(fpr, tpr, alpha=0.12, color=PALETTE["med_blue"])
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.set_title(title)
    ax.legend(loc="lower right")
    fig.tight_layout()
    if save_as:
        fig.savefig(FIGURES_DIR / save_as, bbox_inches="tight")
    plt.show()
    return roc_auc


print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)


Project root: /workspace/projects/titanic-survival-classification
Data directory: /workspace/projects/titanic-survival-classification/data/raw


## Data loading and split

We use a stratified **60 / 20 / 20** train / validation / test split. Validation guides model selection; the test set is used once at the end.


In [2]:
raw = pd.read_csv(DATA_DIR / "data.csv")
y = raw["survived"]
X = raw.drop(columns=["survived"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train
)

print(f"Train: {X_train.shape[0]} | Validation: {X_val.shape[0]} | Test: {X_test.shape[0]}")
print(f"Features: {list(X_train.columns)}")
print("Survival rate (train):", f"{y_train.mean():.3f}")


Train: 600 | Validation: 200 | Test: 200
Features: ['ID', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked', 'home.dest']
Survival rate (train): 0.373


## Exploratory analysis

The target is imbalanced — fewer passengers survived than not. Several columns are categorical with high cardinality (`name`, `ticket`, `cabin`, …), and `age` / `fare` contain missing values.


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

counts = y_train.value_counts().sort_index()
axes[0].bar(
    [CLASS_NAMES[i] for i in counts.index],
    counts.values,
    color=CLASS_COLORS,
    width=0.55,
    edgecolor="white",
)
axes[0].set_title("Class balance (training set)")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 3, str(v), ha="center", color=PALETTE["slate"], fontsize=10)

missing = X_train.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]
axes[1].barh(missing.index, missing.values, color=PALETTE["med_pink"], height=0.6)
axes[1].set_xlabel("Fraction missing")
axes[1].set_title("Missing values by feature")
axes[1].invert_yaxis()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_eda_overview.png", bbox_inches="tight")
plt.show()

display(X_train.describe(include="all").T.head(12))


/tmp/ipykernel_17600/995873281.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID,600.0,NaN,NaN,NaN,499.936667,285.412795,0.0,258.25,487.5,744.5,999.0
pclass,600.0,NaN,NaN,NaN,2.323333,0.836633,1.0,2.0,3.0,3.0,3.0
name,600,600,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sex,600,2,male,400,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,480.0,NaN,NaN,NaN,29.93125,14.386186,0.6667,21.0,28.0,39.0,80.0
sibsp,600.0,NaN,NaN,NaN,0.475,1.044602,0.0,0.0,0.0,1.0,8.0
parch,600.0,NaN,NaN,NaN,0.371667,0.853274,0.0,0.0,0.0,0.0,9.0
ticket,600,502,1601,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fare,600.0,NaN,NaN,NaN,30.003597,44.828722,0.0,7.8958,13.5,30.0,512.3292
cabin,134,107,D,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))

# Survival by sex
sex_surv = pd.crosstab(X_train["sex"], y_train, normalize="index")
sex_surv.plot(
    kind="bar",
    ax=axes[0],
    color=CLASS_COLORS,
    edgecolor="white",
    legend=False,
)
axes[0].set_title("Survival rate by sex")
axes[0].set_xlabel("Sex")
axes[0].set_ylabel("Rate")
axes[0].tick_params(axis="x", rotation=0)

# Survival by passenger class
pclass_surv = pd.crosstab(X_train["pclass"], y_train, normalize="index")
pclass_surv.plot(
    kind="bar",
    ax=axes[1],
    color=CLASS_COLORS,
    edgecolor="white",
    legend=False,
)
axes[1].set_title("Survival rate by class")
axes[1].set_xlabel("Passenger class")
axes[1].tick_params(axis="x", rotation=0)

# Age distribution
for label, color in [(0, CLASS_COLORS[0]), (1, CLASS_COLORS[1])]:
    ages = X_train.loc[y_train.to_numpy() == label, "age"].dropna()
    axes[2].hist(ages, bins=20, alpha=0.55, color=color, label=CLASS_NAMES[label])
axes[2].set_title("Age by outcome")
axes[2].set_xlabel("Age")
axes[2].legend(fontsize=8)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_feature_relationships.png", bbox_inches="tight")
plt.show()


/tmp/ipykernel_17600/2971666873.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Preprocessing

High-cardinality text fields (`name`, `ticket`, `cabin`, `embarked`, `home.dest`) add noise more than signal for these simple models, so we drop them. `sex` is encoded as a binary indicator (`female=1`). Remaining missing numeric values are filled with a sentinel (`-1`) so tree and distance-based models can still use incomplete rows.


In [5]:
COLS_TO_DROP = ["name", "ticket", "cabin", "embarked", "home.dest"]


def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.drop(columns=[c for c in COLS_TO_DROP if c in out.columns])
    if "ID" in out.columns:
        out = out.drop(columns=["ID"])
    out["sex"] = out["sex"].map({"female": 1, "male": 0})
    out = out.fillna(-1)
    return out


X_train_p = preprocess(X_train)
X_val_p = preprocess(X_val)
X_test_p = preprocess(X_test)

print("Features after preprocessing:", list(X_train_p.columns))
print("Missing values left:", int(X_train_p.isna().sum().sum()))
X_train_p.head()


Features after preprocessing: ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']
Missing values left: 0


,pclass,sex,age,sibsp,parch,fare
203,1,1,49.0,1,0,76.7292
308,1,0,48.0,1,0,52.0000
817,1,0,60.0,0,0,26.5500
143,3,0,33.0,0,0,7.8542
923,3,1,-1.0,1,9,69.5500


## Decision Tree

Trees capture non-linear interactions (for example sex × class) without manual feature crosses, but they overfit easily. We tune depth, split criterion, feature subsampling, and leaf size on the validation set.


In [6]:
tree_grid = {
    "max_depth": range(1, 16),
    "criterion": ["entropy", "gini"],
    "max_features": range(1, X_train_p.shape[1] + 1),
    "min_samples_leaf": range(1, 21),
}
tree_combos = list(ParameterGrid(tree_grid))

tree_val_acc, tree_train_acc = [], []
for params in tree_combos:
    clf = DecisionTreeClassifier(**params, random_state=SEED)
    clf.fit(X_train_p, y_train)
    tree_train_acc.append(accuracy_score(y_train, clf.predict(X_train_p)))
    tree_val_acc.append(accuracy_score(y_val, clf.predict(X_val_p)))

best_tree_params = tree_combos[int(np.argmax(tree_val_acc))]
tree_model = DecisionTreeClassifier(**best_tree_params, random_state=SEED)
tree_model.fit(X_train_p, y_train)

print("Best Decision Tree params:", best_tree_params)
print(f"Train accuracy: {accuracy_score(y_train, tree_model.predict(X_train_p)):.4f}")
print(f"Validation accuracy: {accuracy_score(y_val, tree_model.predict(X_val_p)):.4f}")
print(f"Validation F1: {f1_score(y_val, tree_model.predict(X_val_p)):.4f}")

# Accuracy vs depth (best other params fixed for visualization)
depth_vals = sorted(set(p["max_depth"] for p in tree_combos))
val_by_depth, train_by_depth = [], []
for d in depth_vals:
    params = {**best_tree_params, "max_depth": d}
    clf = DecisionTreeClassifier(**params, random_state=SEED)
    clf.fit(X_train_p, y_train)
    train_by_depth.append(accuracy_score(y_train, clf.predict(X_train_p)))
    val_by_depth.append(accuracy_score(y_val, clf.predict(X_val_p)))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(depth_vals, train_by_depth, color=PALETTE["med_pink"], marker="o", label="Train")
ax.plot(depth_vals, val_by_depth, color=PALETTE["med_blue"], marker="o", label="Validation")
ax.set_xlabel("max_depth")
ax.set_ylabel("Accuracy")
ax.set_title("Decision Tree — train vs validation accuracy")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_tree_depth_curve.png", bbox_inches="tight")
plt.show()

plot_confusion(y_val, tree_model.predict(X_val_p), "Decision Tree — validation", "04_cm_tree.png")
tree_auc = plot_roc(
    y_val,
    tree_model.predict_proba(X_val_p)[:, 1],
    "Decision Tree — validation ROC",
    "05_roc_tree.png",
)


Best Decision Tree params: {'criterion': 'entropy', 'max_depth': 6, 'max_features': 4, 'min_samples_leaf': 8}
Train accuracy: 0.8350
Validation accuracy: 0.8750
Validation F1: 0.8252


/tmp/ipykernel_17600/3163320753.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/workspace/common/portfolio_style.py:113: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_17600/4166937745.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## k-Nearest Neighbors

KNN assigns labels from nearby training examples in feature space, which can capture local survival patterns. Distance metrics require scaled features; we use `StandardScaler` fit on the training fold only.


In [7]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_p)
X_val_s = scaler.transform(X_val_p)
X_test_s = scaler.transform(X_test_p)

knn_grid = {
    "n_neighbors": range(1, 21),
    "metric": ["euclidean", "manhattan", "minkowski"],
    "weights": ["uniform", "distance"],
    "algorithm": ["auto", "ball_tree", "kd_tree", "brute"],
}
knn_combos = list(ParameterGrid(knn_grid))

knn_val_acc = []
for params in knn_combos:
    clf = KNeighborsClassifier(**params)
    clf.fit(X_train_s, y_train)
    knn_val_acc.append(accuracy_score(y_val, clf.predict(X_val_s)))

best_knn_params = knn_combos[int(np.argmax(knn_val_acc))]
knn_model = KNeighborsClassifier(**best_knn_params)
knn_model.fit(X_train_s, y_train)

print("Best KNN params:", best_knn_params)
print(f"Train accuracy: {accuracy_score(y_train, knn_model.predict(X_train_s)):.4f}")
print(f"Validation accuracy: {accuracy_score(y_val, knn_model.predict(X_val_s)):.4f}")
print(f"Validation F1: {f1_score(y_val, knn_model.predict(X_val_s)):.4f}")

# Accuracy vs k for the selected metric/weights
ks = list(range(1, 21))
val_by_k, train_by_k = [], []
for k in ks:
    params = {**best_knn_params, "n_neighbors": k}
    clf = KNeighborsClassifier(**params)
    clf.fit(X_train_s, y_train)
    train_by_k.append(accuracy_score(y_train, clf.predict(X_train_s)))
    val_by_k.append(accuracy_score(y_val, clf.predict(X_val_s)))

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.plot(ks, train_by_k, color=PALETTE["med_pink"], marker="o", label="Train")
ax.plot(ks, val_by_k, color=PALETTE["med_blue"], marker="o", label="Validation")
ax.set_xlabel("n_neighbors")
ax.set_ylabel("Accuracy")
ax.set_title("KNN — train vs validation accuracy")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "06_knn_k_curve.png", bbox_inches="tight")
plt.show()

plot_confusion(y_val, knn_model.predict(X_val_s), "KNN — validation", "07_cm_knn.png")
knn_auc = plot_roc(
    y_val,
    knn_model.predict_proba(X_val_s)[:, 1],
    "KNN — validation ROC",
    "08_roc_knn.png",
)


Best KNN params: {'algorithm': 'auto', 'metric': 'manhattan', 'n_neighbors': 10, 'weights': 'uniform'}
Train accuracy: 0.8117
Validation accuracy: 0.8700
Validation F1: 0.8243


/tmp/ipykernel_17600/226523444.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/workspace/common/portfolio_style.py:113: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_17600/4166937745.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Model comparison and final evaluation

We compare validation accuracy, F1, and ROC-AUC, then estimate generalization once on the held-out test set with the selected model.


In [8]:
metrics_table = pd.DataFrame(
    {
        "model": ["Decision Tree", "KNN"],
        "val_accuracy": [
            accuracy_score(y_val, tree_model.predict(X_val_p)),
            accuracy_score(y_val, knn_model.predict(X_val_s)),
        ],
        "val_f1": [
            f1_score(y_val, tree_model.predict(X_val_p)),
            f1_score(y_val, knn_model.predict(X_val_s)),
        ],
        "val_auc": [tree_auc, knn_auc],
    }
).set_index("model")

display(metrics_table.round(4))

fig, ax = plt.subplots(figsize=(7, 3.8))
x = np.arange(len(metrics_table))
width = 0.25
ax.bar(x - width, metrics_table["val_accuracy"], width, label="Accuracy", color=PALETTE["med_blue"])
ax.bar(x, metrics_table["val_f1"], width, label="F1", color=PALETTE["med_pink"])
ax.bar(x + width, metrics_table["val_auc"], width, label="ROC-AUC", color=PALETTE["med_green"])
ax.set_xticks(x)
ax.set_xticklabels(metrics_table.index)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("Score")
ax.set_title("Validation metrics comparison")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "09_model_comparison.png", bbox_inches="tight")
plt.show()

# Prefer higher F1 under class imbalance; break ties with AUC then accuracy
rank = metrics_table.sort_values(["val_f1", "val_auc", "val_accuracy"], ascending=False)
best_name = rank.index[0]
print("Selected model:", best_name)
display(rank.round(4))

if best_name == "Decision Tree":
    final_model = tree_model
    X_test_final = X_test_p
else:
    final_model = knn_model
    X_test_final = X_test_s

y_test_pred = final_model.predict(X_test_final)
print(f"Held-out test accuracy: {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Held-out test F1: {f1_score(y_test, y_test_pred):.4f}")
print(classification_report(y_test, y_test_pred, target_names=[CLASS_NAMES[0], CLASS_NAMES[1]]))
plot_confusion(y_test, y_test_pred, f"{best_name} — test set", "10_cm_final_test.png")


,val_accuracy,val_f1,val_auc
model,,,
Decision Tree,0.875,0.8252,0.9278
KNN,0.870,0.8243,0.9225


/tmp/ipykernel_17600/1470666552.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Selected model: Decision Tree


,val_accuracy,val_f1,val_auc
model,,,
Decision Tree,0.875,0.8252,0.9278
KNN,0.870,0.8243,0.9225


Held-out test accuracy: 0.7700
Held-out test F1: 0.6849
                 precision    recall  f1-score   support

Did not survive       0.81      0.83      0.82       125
       Survived       0.70      0.67      0.68        75

       accuracy                           0.77       200
      macro avg       0.76      0.75      0.75       200
   weighted avg       0.77      0.77      0.77       200



/workspace/common/portfolio_style.py:113: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


array([[104,  21],
       [ 25,  50]])

## Takeaways

1. **Sex and passenger class** are strong univariate signals for survival in this dataset.
2. Dropping high-cardinality identifiers keeps Decision Tree and KNN focused on generalizable structure.
3. Under class imbalance, **F1 and ROC-AUC** complement accuracy when choosing between models.
4. Both models reach competitive validation performance; the final choice is the stronger F1 (with AUC as a tie-breaker) and is reported once on the held-out test set.
